# YOLO26 Fabric - P2 deep tune + per-class confidence
Run top to bottom on Kaggle. Keeps imgsz fixed, uses P2 head, stronger localization loss, explicit optimizer, and heavier fabric-safe augmentation.


In [ ]:
!pip install ultralytics -q


In [ ]:
!pip uninstall -y torch torchvision torchaudio ultralytics
!pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu118
!pip install ultralytics

In [ ]:
# -------------------------
import os
import json
import yaml
import time
import random
import shutil
import platform
from pathlib import Path

import numpy as np
import torch
from ultralytics import YOLO

try:
    import cv2
except Exception:
    cv2 = None

try:
    import pandas as pd
except Exception:
    pd = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Deterministic=False is faster and avoids some CUDA issues on Kaggle.
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

# ============================================================
# MAIN SETTINGS - CHANGE MODEL HERE
# ============================================================
# Đổi model ở đúng dòng BASE_MODEL bên dưới.
# Lưu ý tên đúng của YOLOv10 trong Ultralytics thường là "yolov10s.pt", KHÔNG phải "yolo10s.pt".
# Ví dụ:
#   BASE_MODEL = "yolov10n.pt"  # YOLOv10 nano
#   BASE_MODEL = "yolov10s.pt"  # YOLOv10 small
#   BASE_MODEL = "yolov10m.pt"  # YOLOv10 medium
#   BASE_MODEL = "yolov8s.pt"   # YOLOv8 small
#   BASE_MODEL = "yolov8m.pt"   # YOLOv8 medium
#   BASE_MODEL = "yolo11s.pt"   # YOLO11 small
#   BASE_MODEL = "yolo11m.pt"   # YOLO11 medium
#
# PIPELINE NÀY KHÔNG DÙNG best.pt CŨ.
# Nó train từ BASE_MODEL, train xong mới lấy best.pt mới sinh ra để test.
# PIPELINE NÀY KHÔNG DÙNG best.pt CŨ.
# Nó train từ BASE_MODEL, train xong mới lấy best.pt mới sinh ra để test.
# P2 architecture improves small-object detection without increasing image size.
# It is initialized from base YOLO26s weights, not from any old best.pt.
BASE_MODEL = "yolo26s-p2.yaml"
PRETRAIN_WEIGHTS = "yolo26s.pt"
MODEL_INIT = BASE_MODEL
MODEL_TAG = Path(BASE_MODEL).stem

RUN_NAME = f"{MODEL_TAG}_fabric_p2_deep_tune_300ep_es30"

PROJECT_DIR = Path("/kaggle/working/outputs/fabric_train_then_test")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)


# Keep imgsz fixed. P2 is heavier, so only batch falls back if CUDA OOM.
TRAIN_ATTEMPTS = [
    {"imgsz": 960, "batch": 8},
    {"imgsz": 960, "batch": 6},
    {"imgsz": 960, "batch": 4},
]

# Optional stricter fallback if P2 is still OOM:
# TRAIN_ATTEMPTS = [
#     {"imgsz": 960, "batch": 4},
#     {"imgsz": 960, "batch": 2},
# ]

EPOCHS = 300
PATIENCE = 30
WORKERS = 2

# Heavier localization focus: target low mAP50:95 / AP75 from kq2.json.
BOX_GAIN = 12.0
CLS_GAIN = 0.45
DFL_GAIN = 3.0

# Inference thresholds: predict low first, then filter per class.
PRED_BASE_CONF = 0.20
PRED_NMS_IOU = 0.50
MAX_DET = 1000

CLASS_CONF_BY_NAME = {
    "CHONNGANG": 0.40,
    "DINHDO": 0.40,
    "DINHDO_2": 0.40,
    "SOCMAU": 0.25,
    "SOCXEP": 0.25,
    "XEPGAY": 0.35,
    "XEPLYCUONCAY": 0.30,
    "XEPLYTOVN": 0.25,
    "XEPLY_1": 0.25,
}
DEFAULT_CLASS_CONF = 0.30

SAVE_ANNOTATED_IMAGES = True
SAVE_ANNOTATED_MAX = 300  # set None to save all

print("MODEL_INIT =", MODEL_INIT)
print("PRETRAIN_WEIGHTS =", PRETRAIN_WEIGHTS)
print("RUN_NAME =", RUN_NAME)
print("PROJECT_DIR =", PROJECT_DIR)

In [ ]:
# -------------------------
def get_device():
    if not torch.cuda.is_available():
        print("CUDA not available. Running on CPU.")
        return "cpu"

    try:
        name = torch.cuda.get_device_name(0)
        cc = torch.cuda.get_device_capability(0)
        arch_list = torch.cuda.get_arch_list()
        target_arch = f"sm_{cc[0]}{cc[1]}"
        print(f"Torch: {torch.__version__}, CUDA: {torch.version.cuda}")
        print(f"GPU: {name}, capability: {target_arch}")
        print("Torch compiled arches:", arch_list)

        if arch_list and target_arch not in arch_list:
            raise RuntimeError(
                f"Current torch build does not support GPU arch {target_arch}. "
                "On Kaggle P100, do not install torch cu126. Restart runtime and use Kaggle default torch "
                "or install a cu118 build such as torch==2.4.1+cu118."
            )
        return 0
    except Exception as e:
        raise RuntimeError(f"GPU check failed: {e}")

DEVICE = get_device()

In [ ]:
# -------------------------
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/anhkhoa675/fabric-dataset-ver1/fabric_dataset_yolo"),
    Path("/kaggle/input/fabric-dataset-ver1/fabric_dataset_yolo"),
    Path("/kaggle/input/Fabric_Dataset_ver1/fabric_dataset_yolo"),
    Path("/kaggle/input/fabric-dataset-ver1/Fabric_Dataset_ver1/fabric_dataset_yolo"),
    Path.cwd() / "data/processed/fabric_dataset_yolo",
    Path.cwd().parent / "data/processed/fabric_dataset_yolo",
]

dataset_root = next((p for p in DATASET_CANDIDATES if (p / "dataset.yaml").exists()), None)
if dataset_root is None:
    checked = "\n".join(str(p) for p in DATASET_CANDIDATES)
    raise FileNotFoundError("Cannot find fabric_dataset_yolo/dataset.yaml. Checked:\n" + checked)

source_yaml_path = dataset_root / "dataset.yaml"
with open(source_yaml_path, "r", encoding="utf-8") as f:
    source_data = yaml.safe_load(f)

names_raw = source_data.get("names", {})
if isinstance(names_raw, dict):
    class_names = [names_raw[k] for k in sorted(names_raw, key=lambda x: int(x))]
elif isinstance(names_raw, list):
    class_names = names_raw
else:
    raise ValueError("dataset.yaml must contain names as list or dict")

def split_image_dir(split_name):
    value = source_data.get(split_name, f"images/{split_name}")
    split_path = Path(value)
    if not split_path.is_absolute():
        split_path = dataset_root / split_path
    return str(split_path)

data_fixed = {
    "train": split_image_dir("train"),
    "val": split_image_dir("val"),
    "test": split_image_dir("test"),
    "nc": len(class_names),
    "names": class_names,
}

DATA_YAML = Path("/kaggle/working/data.yaml")
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.dump(data_fixed, f, sort_keys=False, allow_unicode=True)

print("dataset_root =", dataset_root)
print("DATA_YAML =", DATA_YAML)
print(yaml.dump(data_fixed, sort_keys=False, allow_unicode=True))

for split in ["train", "val", "test"]:
    p = Path(data_fixed[split])
    print(split, p, "exists=", p.exists())

CLASS_CONF = {i: CLASS_CONF_BY_NAME.get(name, DEFAULT_CLASS_CONF) for i, name in enumerate(class_names)}
print("CLASS_CONF:", {class_names[i]: v for i, v in CLASS_CONF.items()})

In [ ]:
# -------------------------
def is_cuda_oom(exc):
    msg = str(exc).lower()
    return "out of memory" in msg or "cuda oom" in msg or "cublas" in msg or "cuda" in msg

def build_train_model():
    model = YOLO(MODEL_INIT)
    if PRETRAIN_WEIGHTS:
        print(f"Loading transfer weights: {PRETRAIN_WEIGHTS}")
        model = model.load(PRETRAIN_WEIGHTS)
    return model

def train_once(imgsz, batch):
    print("\n" + "=" * 70)
    print(f"Training attempt: model={MODEL_INIT}, pretrain={PRETRAIN_WEIGHTS}, imgsz={imgsz}, batch={batch}")
    print("=" * 70)

    model = build_train_model()
    results = model.train(
        data=str(DATA_YAML),
        project=str(PROJECT_DIR),
        name=RUN_NAME,
        imgsz=imgsz,
        batch=batch,
        epochs=EPOCHS,
        patience=PATIENCE,
        device=DEVICE,
        workers=WORKERS,
        seed=SEED,
        deterministic=False,
        pretrained=False,
        exist_ok=True,
        save=True,
        plots=True,

        # Tuned loss weights
        box=BOX_GAIN,
        cls=CLS_GAIN,
        dfl=DFL_GAIN,

        # Scheduler / optimization
        optimizer="AdamW",
        lr0=0.001,
        lrf=0.01,
        warmup_epochs=8,
        cos_lr=True,
        close_mosaic=120,
        weight_decay=0.005,

        # Stronger but fabric-safe augmentation. Geometry stays mild for tighter boxes.
        degrees=1.0,
        translate=0.02,
        scale=0.15,
        shear=0.0,
        perspective=0.0,
        mosaic=0.30,
        mixup=0.03,
        cutmix=0.05,
        copy_paste=0.0,
        erasing=0.0,
        fliplr=0.5,
        flipud=0.0,
    )
    return model, results

last_exc = None
train_model = None
train_results = None
FINAL_IMGSZ = None
FINAL_BATCH = None

for attempt in TRAIN_ATTEMPTS:
    try:
        train_model, train_results = train_once(attempt["imgsz"], attempt["batch"])
        FINAL_IMGSZ = attempt["imgsz"]
        FINAL_BATCH = attempt["batch"]
        break
    except Exception as e:
        last_exc = e
        print("Training attempt failed:", repr(e))
        if DEVICE != "cpu" and torch.cuda.is_available():
            torch.cuda.empty_cache()
        if not is_cuda_oom(e):
            raise
        print("CUDA issue/OOM detected. Trying smaller batch/imgsz...")

if train_results is None:
    raise RuntimeError(f"All training attempts failed. Last error: {last_exc}")

# IMPORTANT: this is the best.pt created by the training run above.
best_model_path = str(Path(train_results.save_dir) / "weights" / "best.pt")
last_model_path = str(Path(train_results.save_dir) / "weights" / "last.pt")
if not Path(best_model_path).exists():
    raise FileNotFoundError(f"Cannot find NEW trained best.pt: {best_model_path}")

print("\nTRAIN DONE")
print("NEW best_model_path =", best_model_path)
print("last_model_path =", last_model_path)
print("FINAL_IMGSZ =", FINAL_IMGSZ, "FINAL_BATCH =", FINAL_BATCH)

# Copy key artifacts to /kaggle/working for easy download.
BEST_ALIAS = Path(f"/kaggle/working/best_{MODEL_TAG}_fabric_img_loss_tuned.pt")
shutil.copy2(best_model_path, BEST_ALIAS)
print("Copied NEW best model to", BEST_ALIAS)

train_results_csv = Path(train_results.save_dir) / "results.csv"
if train_results_csv.exists():
    shutil.copy2(train_results_csv, "/kaggle/working/history.csv")
    print("Copied history to /kaggle/working/history.csv")

# Free training model memory before test/predict.
del train_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:

assert best_model_path is not None and Path(best_model_path).exists(), f"Không tìm thấy best.pt mới: {best_model_path}"

# -------------------------
print("\n" + "=" * 70)
print("TRAIN ĐÃ XONG -> BẮT ĐẦU TEST NEW TRAINED BEST.PT")
print("Loading:", best_model_path)
print("=" * 70)

test_model = YOLO(best_model_path)
test_results = test_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=FINAL_IMGSZ,
    conf=0.001,      # low conf for proper mAP curve
    iou=PRED_NMS_IOU,
    batch=max(1, FINAL_BATCH),
    device=DEVICE,
    workers=WORKERS,
    project=str(PROJECT_DIR),
    name=f"{RUN_NAME}_test_eval",
    exist_ok=True,
    verbose=True,
    plots=True,
)

ultralytics_metrics = {
    "mAP_0.5": round(float(test_results.box.map50), 4),
    "mAP_0.5:0.95": round(float(test_results.box.map), 4),
    "precision": round(float(test_results.box.mp), 4),
    "recall": round(float(test_results.box.mr), 4),
    "f1_score": round(float(2 * test_results.box.mp * test_results.box.mr / (test_results.box.mp + test_results.box.mr + 1e-16)), 4),
}
print("Ultralytics TEST metrics:", ultralytics_metrics)

In [ ]:
# -------------------------
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def list_images(folder):
    folder = Path(folder)
    return sorted([p for p in folder.rglob("*") if p.suffix.lower() in IMG_EXTS])

def label_path_for_image(img_path):
    s = str(img_path)
    if "/images/" in s:
        return Path(s.replace("/images/", "/labels/")).with_suffix(".txt")
    return img_path.with_suffix(".txt")

def load_yolo_label(label_path, img_w, img_h):
    labels = []
    label_path = Path(label_path)
    if not label_path.exists():
        return labels
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cid = int(float(parts[0]))
            cx, cy, bw, bh = map(float, parts[1:5])
            x1 = (cx - bw / 2) * img_w
            y1 = (cy - bh / 2) * img_h
            x2 = (cx + bw / 2) * img_w
            y2 = (cy + bh / 2) * img_h
            labels.append({
                "class_id": cid,
                "class_name": class_names[cid] if cid < len(class_names) else str(cid),
                "bbox": [float(x1), float(y1), float(x2), float(y2)],
            })
    return labels

def box_iou_single(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    if inter <= 0:
        return 0.0
    a1 = max(0.0, box1[2] - box1[0]) * max(0.0, box1[3] - box1[1])
    a2 = max(0.0, box2[2] - box2[0]) * max(0.0, box2[3] - box2[1])
    denom = a1 + a2 - inter
    return float(inter / denom) if denom > 0 else 0.0

def compute_ap(recall, precision):
    if len(recall) == 0:
        return 0.0
    mrec = np.concatenate(([0.0], np.asarray(recall), [1.0]))
    mpre = np.concatenate(([0.0], np.asarray(precision), [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))

def get_object_size_category(bbox):
    area = max(0.0, bbox[2] - bbox[0]) * max(0.0, bbox[3] - bbox[1])
    if area < 32 * 32:
        return "small"
    if area < 96 * 96:
        return "medium"
    return "large"

def class_color(cid):
    rng = np.random.default_rng(cid + 12345)
    return tuple(int(x) for x in rng.integers(40, 230, size=3))

def draw_detections(img_path, detections, out_path):
    if cv2 is None:
        return
    img = cv2.imread(str(img_path))
    if img is None:
        return
    for det in detections:
        cid = det["class_id"]
        conf = det["confidence"]
        name = det["class_name"]
        x1, y1, x2, y2 = [int(round(v)) for v in det["bbox"]]
        color = class_color(cid)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        txt = f"{name} {conf:.2f}"
        cv2.putText(img, txt, (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), img)

def predict_split_with_per_class_conf(split="test"):
    image_files = list_images(data_fixed[split])
    out_dir = Path("/kaggle/working/perclass_conf_outputs") / MODEL_TAG / split
    ann_dir = out_dir / "annotated"
    label_dir = out_dir / "labels_txt"
    out_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    all_records = []
    flat_predictions = []
    annotated_count = 0

    print(f"Predicting NEW best.pt on split={split}, images={len(image_files)}, base_conf={PRED_BASE_CONF}, nms_iou={PRED_NMS_IOU}")

    for img_idx, img_path in enumerate(image_files, start=1):
        preds = test_model.predict(
            source=str(img_path),
            imgsz=FINAL_IMGSZ,
            conf=PRED_BASE_CONF,
            iou=PRED_NMS_IOU,
            max_det=MAX_DET,
            device=DEVICE,
            verbose=False,
        )
        result = preds[0]
        img_h, img_w = result.orig_shape
        gt = load_yolo_label(label_path_for_image(img_path), img_w, img_h)

        detections = []
        boxes = result.boxes
        if boxes is not None and len(boxes) > 0:
            for box in boxes:
                cid = int(box.cls.item())
                conf = float(box.conf.item())
                threshold = CLASS_CONF.get(cid, DEFAULT_CLASS_CONF)
                if conf < threshold:
                    continue
                x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].tolist()]
                cls_name = class_names[cid] if cid < len(class_names) else str(cid)
                det = {
                    "class_id": cid,
                    "class_name": cls_name,
                    "confidence": conf,
                    "threshold_used": threshold,
                    "bbox": [x1, y1, x2, y2],
                }
                detections.append(det)
                flat_predictions.append({
                    "image_id": img_path.stem,
                    "file_name": img_path.name,
                    "class_id": cid,
                    "class_name": cls_name,
                    "confidence": round(conf, 6),
                    "threshold_used": threshold,
                    "bbox_xyxy": [round(x1, 3), round(y1, 3), round(x2, 3), round(y2, 3)],
                    "bbox_xywh": [round(x1, 3), round(y1, 3), round(x2 - x1, 3), round(y2 - y1, 3)],
                })

        # Save YOLO-style txt predictions with confidence: cls cx cy w h conf
        txt_path = label_dir / f"{img_path.stem}.txt"
        with open(txt_path, "w", encoding="utf-8") as f:
            for det in detections:
                x1, y1, x2, y2 = det["bbox"]
                cx = ((x1 + x2) / 2) / img_w
                cy = ((y1 + y2) / 2) / img_h
                bw = (x2 - x1) / img_w
                bh = (y2 - y1) / img_h
                f.write(f"{det['class_id']} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f} {det['confidence']:.6f}\n")

        if SAVE_ANNOTATED_IMAGES and (SAVE_ANNOTATED_MAX is None or annotated_count < SAVE_ANNOTATED_MAX):
            draw_detections(img_path, detections, ann_dir / img_path.name)
            annotated_count += 1

        all_records.append({
            "image_name": img_path.name,
            "image_path": str(img_path),
            "ground_truth": gt,
            "detections": detections,
        })

        if img_idx % 50 == 0:
            print(f"  processed {img_idx}/{len(image_files)}")

    records_path = out_dir / "records_per_image.json"
    preds_path = out_dir / "predictions_perclass_conf.json"
    with open(records_path, "w", encoding="utf-8") as f:
        json.dump(all_records, f, indent=2, ensure_ascii=False)
    with open(preds_path, "w", encoding="utf-8") as f:
        json.dump(flat_predictions, f, indent=2, ensure_ascii=False)

    print("Saved records to", records_path)
    print("Saved flat predictions to", preds_path)
    return all_records, flat_predictions, out_dir

def compute_class_ap_at_iou(all_records, cls_id, iou_thresh):
    scores, tp, fp = [], [], []
    n_gt = 0
    for rec in all_records:
        gts = [g for g in rec["ground_truth"] if g["class_id"] == cls_id]
        dets = sorted([d for d in rec["detections"] if d["class_id"] == cls_id], key=lambda x: -x["confidence"])
        n_gt += len(gts)
        matched = [False] * len(gts)
        for det in dets:
            scores.append(det["confidence"])
            best_iou, best_j = 0.0, -1
            for j, gt in enumerate(gts):
                iou = box_iou_single(det["bbox"], gt["bbox"])
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_j >= 0 and best_iou >= iou_thresh and not matched[best_j]:
                tp.append(1)
                fp.append(0)
                matched[best_j] = True
            else:
                tp.append(0)
                fp.append(1)

    if not scores or n_gt == 0:
        return 0.0
    order = np.argsort(scores)[::-1]
    tp_cum = np.cumsum(np.asarray(tp)[order])
    fp_cum = np.cumsum(np.asarray(fp)[order])
    recall = tp_cum / (n_gt + 1e-16)
    precision = tp_cum / (tp_cum + fp_cum + 1e-16)
    return compute_ap(recall, precision)

def compute_per_class_metrics(all_records, class_names, iou_thresh=0.5):
    rows = []
    for cls_id, cls_name in enumerate(class_names):
        scores, tp, fp = [], [], []
        n_gt = 0
        for rec in all_records:
            gts = [g for g in rec["ground_truth"] if g["class_id"] == cls_id]
            dets = sorted([d for d in rec["detections"] if d["class_id"] == cls_id], key=lambda x: -x["confidence"])
            n_gt += len(gts)
            matched = [False] * len(gts)
            for det in dets:
                scores.append(det["confidence"])
                best_iou, best_j = 0.0, -1
                for j, gt in enumerate(gts):
                    iou = box_iou_single(det["bbox"], gt["bbox"])
                    if iou > best_iou:
                        best_iou, best_j = iou, j
                if best_j >= 0 and best_iou >= iou_thresh and not matched[best_j]:
                    tp.append(1)
                    fp.append(0)
                    matched[best_j] = True
                else:
                    tp.append(0)
                    fp.append(1)

        n_pred = len(scores)
        if n_pred == 0:
            rows.append({
                "class_id": cls_id,
                "class_name": cls_name,
                "conf_threshold": CLASS_CONF.get(cls_id, DEFAULT_CLASS_CONF),
                "num_gt": n_gt,
                "num_pred": 0,
                "tp": 0,
                "fp": 0,
                "fn": n_gt,
                "precision": 0.0,
                "recall": 0.0,
                "f1_score": 0.0,
                "ap_0.5": 0.0,
                "ap_0.75": 0.0,
                "ap_0.5:0.95": 0.0,
            })
            continue

        order = np.argsort(scores)[::-1]
        tp_arr = np.asarray(tp)[order]
        fp_arr = np.asarray(fp)[order]
        tp_cum = np.cumsum(tp_arr)
        fp_cum = np.cumsum(fp_arr)
        recall_curve = tp_cum / (n_gt + 1e-16)
        precision_curve = tp_cum / (tp_cum + fp_cum + 1e-16)

        total_tp = int(tp_cum[-1])
        total_fp = int(fp_cum[-1])
        total_fn = int(n_gt - total_tp)
        precision = total_tp / (total_tp + total_fp + 1e-16)
        recall = total_tp / (n_gt + 1e-16)
        f1 = 2 * precision * recall / (precision + recall + 1e-16)
        ap05 = compute_ap(recall_curve, precision_curve)
        ap075 = compute_class_ap_at_iou(all_records, cls_id, 0.75)
        ap5095 = float(np.mean([compute_class_ap_at_iou(all_records, cls_id, t) for t in np.arange(0.5, 1.0, 0.05)]))

        rows.append({
            "class_id": cls_id,
            "class_name": cls_name,
            "conf_threshold": CLASS_CONF.get(cls_id, DEFAULT_CLASS_CONF),
            "num_gt": int(n_gt),
            "num_pred": int(n_pred),
            "tp": total_tp,
            "fp": total_fp,
            "fn": total_fn,
            "precision": round(float(precision), 4),
            "recall": round(float(recall), 4),
            "f1_score": round(float(f1), 4),
            "ap_0.5": round(float(ap05), 4),
            "ap_0.75": round(float(ap075), 4),
            "ap_0.5:0.95": round(float(ap5095), 4),
        })
    return rows

def compute_size_based_metrics(all_records, iou_thresh=0.5):
    out = {}
    for size in ["small", "medium", "large"]:
        scores, tp, fp = [], [], []
        n_gt = 0
        for rec in all_records:
            gts = [g for g in rec["ground_truth"] if get_object_size_category(g["bbox"]) == size]
            dets = sorted([d for d in rec["detections"] if get_object_size_category(d["bbox"]) == size], key=lambda x: -x["confidence"])
            n_gt += len(gts)
            matched = [False] * len(gts)
            for det in dets:
                scores.append(det["confidence"])
                best_iou, best_j = 0.0, -1
                for j, gt in enumerate(gts):
                    if det["class_id"] != gt["class_id"]:
                        continue
                    iou = box_iou_single(det["bbox"], gt["bbox"])
                    if iou > best_iou:
                        best_iou, best_j = iou, j
                if best_j >= 0 and best_iou >= iou_thresh and not matched[best_j]:
                    tp.append(1)
                    fp.append(0)
                    matched[best_j] = True
                else:
                    tp.append(0)
                    fp.append(1)

        n_pred = len(scores)
        if n_pred == 0:
            out[size] = {"num_gt": int(n_gt), "num_pred": 0, "tp": 0, "fp": 0, "fn": int(n_gt), "precision": 0.0, "recall": 0.0, "f1_score": 0.0, "ap_0.5": 0.0}
            continue

        order = np.argsort(scores)[::-1]
        tp_arr = np.asarray(tp)[order]
        fp_arr = np.asarray(fp)[order]
        tp_cum = np.cumsum(tp_arr)
        fp_cum = np.cumsum(fp_arr)
        recall_curve = tp_cum / (n_gt + 1e-16)
        precision_curve = tp_cum / (tp_cum + fp_cum + 1e-16)
        total_tp = int(tp_cum[-1])
        total_fp = int(fp_cum[-1])
        total_fn = int(n_gt - total_tp)
        precision = total_tp / (total_tp + total_fp + 1e-16)
        recall = total_tp / (n_gt + 1e-16)
        f1 = 2 * precision * recall / (precision + recall + 1e-16)
        ap05 = compute_ap(recall_curve, precision_curve)
        out[size] = {
            "num_gt": int(n_gt),
            "num_pred": int(n_pred),
            "tp": total_tp,
            "fp": total_fp,
            "fn": total_fn,
            "precision": round(float(precision), 4),
            "recall": round(float(recall), 4),
            "f1_score": round(float(f1), 4),
            "ap_0.5": round(float(ap05), 4),
        }
    return out

def summarize_overall(per_class_rows):
    total_gt = sum(r["num_gt"] for r in per_class_rows)
    total_pred = sum(r["num_pred"] for r in per_class_rows)
    total_tp = sum(r["tp"] for r in per_class_rows)
    total_fp = sum(r["fp"] for r in per_class_rows)
    total_fn = sum(r["fn"] for r in per_class_rows)
    precision = total_tp / (total_tp + total_fp + 1e-16)
    recall = total_tp / (total_gt + 1e-16)
    f1 = 2 * precision * recall / (precision + recall + 1e-16)
    valid_rows = [r for r in per_class_rows if r["num_gt"] > 0]
    map50 = float(np.mean([r["ap_0.5"] for r in valid_rows])) if valid_rows else 0.0
    map5095 = float(np.mean([r["ap_0.5:0.95"] for r in valid_rows])) if valid_rows else 0.0
    return {
        "num_gt": int(total_gt),
        "num_pred": int(total_pred),
        "tp": int(total_tp),
        "fp": int(total_fp),
        "fn": int(total_fn),
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1_score": round(float(f1), 4),
        "mAP_0.5": round(float(map50), 4),
        "mAP_0.5:0.95": round(float(map5095), 4),
    }

In [ ]:
# -------------------------
test_records, test_flat_predictions, perclass_out_dir = predict_split_with_per_class_conf("test")
per_class_metrics = compute_per_class_metrics(test_records, class_names, iou_thresh=0.5)
size_metrics = compute_size_based_metrics(test_records, iou_thresh=0.5)
overall_custom_metrics = summarize_overall(per_class_metrics)

print("\nCustom metrics after per-class confidence filtering:")
print(overall_custom_metrics)

if pd is not None:
    display_cols = ["class_id", "class_name", "conf_threshold", "num_gt", "num_pred", "tp", "fp", "fn", "precision", "recall", "f1_score", "ap_0.5", "ap_0.5:0.95"]
    df_pc = pd.DataFrame(per_class_metrics)[display_cols]
    print("\nPer-class metrics:")
    display(df_pc)
    print("\nSize metrics:")
    display(pd.DataFrame(size_metrics).T)
else:
    print(json.dumps(per_class_metrics, indent=2, ensure_ascii=False))
    print(json.dumps(size_metrics, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# FULL YOLO DETECTION EVALUATION
# Export:
#   - predictions_full_eval.json
#   - eval_summary.json
#   - confusion_matrix.csv
#   - confusion_matrix.png
#
# Run this AFTER:
#   test_records, test_flat_predictions, perclass_out_dir = predict_split_with_per_class_conf("test")
# ============================================================

import json
import csv
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict

# =========================
# CONFIG
# =========================
EVAL_IOU_THRESH = 0.50
INCLUDE_BACKGROUND_IN_CM = True

# Nếu notebook của bạn đã có class_names thì dùng luôn.
# class_names nên là list:
# ["CHONNGANG", "DINHDO", "DINHDO_2", "SOCMAU", "SOCXEP", "XEPGAY", "XEPLYCUONCAY", "XEPLYTOVN", "XEPLY_1"]


# =========================
# HELPER FUNCTIONS
# =========================
def xyxy_iou(box_a, box_b):
    """
    box_a, box_b: [x1, y1, x2, y2]
    Return IoU.
    """
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = area_a + area_b - inter_area
    if union <= 0:
        return 0.0

    return inter_area / union


def normalize_gt_item(g, class_names):
    """
    Accepts possible GT formats:
      {"class_id": ..., "class_name": ..., "bbox": [...]}
      {"class_id": ..., "class_name": ..., "box": [...]}
    """
    cls_id = int(g["class_id"])
    cls_name = g.get("class_name", class_names[cls_id])

    box = g.get("box", None)
    if box is None:
        box = g.get("bbox", None)
    if box is None:
        box = g.get("bbox_xyxy", None)

    return {
        "class_id": cls_id,
        "class_name": cls_name,
        "box": [float(x) for x in box],
    }


def normalize_pred_item(p, class_names):
    """
    Accepts possible prediction formats:
      {"class_id": ..., "class_name": ..., "confidence": ..., "bbox": [...]}
      {"class_id": ..., "class_name": ..., "confidence": ..., "box": [...]}
      {"class_id": ..., "class_name": ..., "confidence": ..., "bbox_xyxy": [...]}
    """
    cls_id = int(p["class_id"])
    cls_name = p.get("class_name", class_names[cls_id])

    box = p.get("box", None)
    if box is None:
        box = p.get("bbox", None)
    if box is None:
        box = p.get("bbox_xyxy", None)

    return {
        "class_id": cls_id,
        "class_name": cls_name,
        "confidence": float(p.get("confidence", 0.0)),
        "box": [float(x) for x in box],
    }


def get_record_image_name(rec):
    return (
        rec.get("image_name")
        or rec.get("file_name")
        or rec.get("image_id")
        or "unknown_image"
    )


def get_record_gts(rec, class_names):
    """
    Supports:
      rec["ground_truth"]
      rec["ground_truths"]
      rec["gts"]
    """
    raw_gts = (
        rec.get("ground_truths", None)
        or rec.get("ground_truth", None)
        or rec.get("gts", None)
        or []
    )

    return [normalize_gt_item(g, class_names) for g in raw_gts]


def get_record_preds(rec, class_names):
    """
    Supports:
      rec["detections"]
      rec["predictions"]
      rec["preds"]
    """
    raw_preds = (
        rec.get("predictions", None)
        or rec.get("detections", None)
        or rec.get("preds", None)
        or []
    )

    return [normalize_pred_item(p, class_names) for p in raw_preds]


# =========================
# MAIN EXPORT FUNCTION
# =========================
def export_full_detection_eval(
    all_records,
    class_names,
    out_dir,
    iou_thresh=0.5,
    include_background=True,
    output_json_name="predictions_full_eval.json",
):
    """
    all_records:
      output từ predict_split_with_per_class_conf("test")

    Output JSON format:
    {
      "meta": {...},
      "per_image": [
        {
          "image_name": ...,
          "ground_truths": [...],
          "predictions": [...]
        }
      ],
      "records": [
        {
          "image_name": ...,
          "match_type": "TP/FP/FN/WRONG_CLASS",
          "gt_class_name": ...,
          "pred_class_name": ...,
          "confidence": ...,
          "iou": ...
        }
      ],
      "summary": {...}
    }
    """

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    class_names = list(class_names)

    labels = list(class_names)
    if include_background:
        labels = labels + ["background"]

    cm = {gt_name: Counter() for gt_name in labels}
    eval_records = []
    per_image = []

    for rec in all_records:
        image_name = get_record_image_name(rec)

        ground_truths = get_record_gts(rec, class_names)
        predictions = get_record_preds(rec, class_names)

        per_image.append({
            "image_name": image_name,
            "ground_truths": ground_truths,
            "predictions": predictions,
        })

        matched_gt_indices = set()

        # Sort predictions by confidence high -> low
        pred_indices = sorted(
            range(len(predictions)),
            key=lambda i: predictions[i]["confidence"],
            reverse=True
        )

        for pred_idx in pred_indices:
            pred = predictions[pred_idx]

            best_iou = 0.0
            best_gt_idx = -1

            # Find best unmatched GT by IoU
            for gt_idx, gt in enumerate(ground_truths):
                if gt_idx in matched_gt_indices:
                    continue

                iou = xyxy_iou(pred["box"], gt["box"])

                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx

            # No GT matched enough IoU => FP
            if best_gt_idx < 0 or best_iou < iou_thresh:
                eval_records.append({
                    "image_name": image_name,
                    "match_type": "FP",
                    "gt_class_id": None,
                    "gt_class_name": "background",
                    "pred_class_id": pred["class_id"],
                    "pred_class_name": pred["class_name"],
                    "confidence": pred["confidence"],
                    "iou": float(best_iou),
                    "gt_box": None,
                    "pred_box": pred["box"],
                })

                if include_background:
                    cm["background"][pred["class_name"]] += 1

                continue

            # Matched a GT
            gt = ground_truths[best_gt_idx]
            matched_gt_indices.add(best_gt_idx)

            if pred["class_id"] == gt["class_id"]:
                match_type = "TP"
            else:
                match_type = "WRONG_CLASS"

            eval_records.append({
                "image_name": image_name,
                "match_type": match_type,
                "gt_class_id": gt["class_id"],
                "gt_class_name": gt["class_name"],
                "pred_class_id": pred["class_id"],
                "pred_class_name": pred["class_name"],
                "confidence": pred["confidence"],
                "iou": float(best_iou),
                "gt_box": gt["box"],
                "pred_box": pred["box"],
            })

            cm[gt["class_name"]][pred["class_name"]] += 1

        # Unmatched GT => FN
        for gt_idx, gt in enumerate(ground_truths):
            if gt_idx not in matched_gt_indices:
                eval_records.append({
                    "image_name": image_name,
                    "match_type": "FN",
                    "gt_class_id": gt["class_id"],
                    "gt_class_name": gt["class_name"],
                    "pred_class_id": None,
                    "pred_class_name": "background",
                    "confidence": None,
                    "iou": None,
                    "gt_box": gt["box"],
                    "pred_box": None,
                })

                if include_background:
                    cm[gt["class_name"]]["background"] += 1

    # =========================
    # SUMMARY METRICS
    # =========================
    match_counts = Counter(r["match_type"] for r in eval_records)

    tp = match_counts.get("TP", 0)
    fp = match_counts.get("FP", 0)
    fn = match_counts.get("FN", 0)
    wrong = match_counts.get("WRONG_CLASS", 0)

    # Với object detection:
    # WRONG_CLASS thường tính vừa làm sai class dự đoán, vừa miss class thật.
    precision = tp / (tp + fp + wrong + 1e-16)
    recall = tp / (tp + fn + wrong + 1e-16)
    f1 = 2 * precision * recall / (precision + recall + 1e-16)

    total_gt = sum(1 for r in eval_records if r["gt_class_name"] != "background")
    total_pred = sum(1 for r in eval_records if r["pred_class_name"] != "background")

    summary = {
        "iou_threshold": float(iou_thresh),
        "num_images": int(len(all_records)),
        "num_eval_records": int(len(eval_records)),
        "num_ground_truth_objects": int(total_gt),
        "num_predicted_objects": int(total_pred),
        "match_counts": dict(match_counts),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "wrong_class": int(wrong),
        "precision": round(float(precision), 6),
        "recall": round(float(recall), 6),
        "f1_score": round(float(f1), 6),
    }

    output = {
        "meta": {
            "iou_threshold": float(iou_thresh),
            "num_images": int(len(all_records)),
            "class_names": {str(i): name for i, name in enumerate(class_names)},
            "include_background_in_confusion_matrix": bool(include_background),
        },
        "per_image": per_image,
        "records": eval_records,
        "summary": summary,
    }

    # =========================
    # SAVE JSON
    # =========================
    json_path = out_dir / output_json_name
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    summary_path = out_dir / "eval_summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    # =========================
    # SAVE CONFUSION MATRIX CSV
    # =========================
    cm_csv_path = out_dir / "confusion_matrix.csv"

    with open(cm_csv_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["actual\\predicted"] + labels)

        for gt_name in labels:
            row = [gt_name] + [cm[gt_name][pred_name] for pred_name in labels]
            writer.writerow(row)

    # =========================
    # DRAW CONFUSION MATRIX PNG
    # =========================
    cm_array = np.array(
        [[cm[gt_name][pred_name] for pred_name in labels] for gt_name in labels],
        dtype=int
    )

    fig, ax = plt.subplots(figsize=(13, 11))
    im = ax.imshow(cm_array)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))

    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class / Ground truth")
    ax.set_title(f"Confusion Matrix - IoU >= {iou_thresh}")

    for i in range(len(labels)):
        for j in range(len(labels)):
            value = cm_array[i, j]
            if value > 0:
                ax.text(j, i, str(value), ha="center", va="center")

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()

    cm_png_path = out_dir / "confusion_matrix.png"
    fig.savefig(cm_png_path, dpi=200, bbox_inches="tight")
    plt.show()

    # =========================
    # SAVE PER-CLASS METRICS
    # =========================
    per_class_rows = []

    for cls_name in class_names:
        cls_tp = sum(
            1 for r in eval_records
            if r["match_type"] == "TP"
            and r["gt_class_name"] == cls_name
            and r["pred_class_name"] == cls_name
        )

        cls_fp = sum(
            1 for r in eval_records
            if r["pred_class_name"] == cls_name
            and r["match_type"] in ["FP", "WRONG_CLASS"]
        )

        cls_fn = sum(
            1 for r in eval_records
            if r["gt_class_name"] == cls_name
            and r["match_type"] in ["FN", "WRONG_CLASS"]
        )

        cls_precision = cls_tp / (cls_tp + cls_fp + 1e-16)
        cls_recall = cls_tp / (cls_tp + cls_fn + 1e-16)
        cls_f1 = 2 * cls_precision * cls_recall / (cls_precision + cls_recall + 1e-16)

        per_class_rows.append({
            "class_name": cls_name,
            "tp": int(cls_tp),
            "fp": int(cls_fp),
            "fn": int(cls_fn),
            "precision": round(float(cls_precision), 6),
            "recall": round(float(cls_recall), 6),
            "f1_score": round(float(cls_f1), 6),
        })

    per_class_csv_path = out_dir / "per_class_metrics.csv"
    with open(per_class_csv_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["class_name", "tp", "fp", "fn", "precision", "recall", "f1_score"]
        )
        writer.writeheader()
        writer.writerows(per_class_rows)

    # =========================
    # PRINT RESULT
    # =========================
    print("=" * 80)
    print("FULL DETECTION EVALUATION DONE")
    print("=" * 80)
    print("Saved:", json_path)
    print("Saved:", summary_path)
    print("Saved:", cm_csv_path)
    print("Saved:", cm_png_path)
    print("Saved:", per_class_csv_path)
    print()
    print("Summary:")
    print(json.dumps(summary, indent=2, ensure_ascii=False))

    return {
        "output": output,
        "summary": summary,
        "json_path": json_path,
        "summary_path": summary_path,
        "confusion_matrix_csv": cm_csv_path,
        "confusion_matrix_png": cm_png_path,
        "per_class_metrics_csv": per_class_csv_path,
    }


# =========================
# RUN EXPORT
# =========================
full_eval_result = export_full_detection_eval(
    all_records=test_records,
    class_names=class_names,
    out_dir=perclass_out_dir,
    iou_thresh=EVAL_IOU_THRESH,
    include_background=INCLUDE_BACKGROUND_IN_CM,
    output_json_name="predictions_full_eval.json",
)

In [ ]:
# -------------------------
def benchmark_model(model_path, image_files, device, n_bench=50):
    n = min(n_bench, len(image_files))
    if n == 0:
        return {"fps": 0.0, "latency_ms": 0.0, "device": str(device)}
    bench_model = YOLO(model_path)
    for _ in range(3):
        bench_model.predict(str(image_files[0]), imgsz=FINAL_IMGSZ, conf=0.25, iou=PRED_NMS_IOU, device=device, verbose=False)
    t0 = time.perf_counter()
    for img in image_files[:n]:
        bench_model.predict(str(img), imgsz=FINAL_IMGSZ, conf=0.25, iou=PRED_NMS_IOU, device=device, verbose=False)
    elapsed = time.perf_counter() - t0
    avg_ms = elapsed / n * 1000
    fps = 1000 / avg_ms if avg_ms > 0 else 0.0
    if device != "cpu" and torch.cuda.is_available():
        device_label = torch.cuda.get_device_name(0)
    else:
        device_label = platform.processor() or "CPU"
    del bench_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"fps": round(float(fps), 4), "latency_ms": round(float(avg_ms), 4), "device": device_label}

test_images_list = list_images(data_fixed["test"])
speed_info = {"cpu": benchmark_model(best_model_path, test_images_list, "cpu", n_bench=32)}
if torch.cuda.is_available() and DEVICE != "cpu":
    speed_info["gpu"] = benchmark_model(best_model_path, test_images_list, 0, n_bench=32)
print("Speed:", speed_info) 

In [ ]:
# -------------------------
# FINAL OUTPUT JSON + ZIP
# -------------------------

FINAL_JSON = Path(f"/kaggle/working/final_results_{MODEL_TAG}_perclass_conf.json")

# Các file full evaluation được tạo từ cell export_full_detection_eval()
FULL_EVAL_JSON = perclass_out_dir / "predictions_full_eval.json"
EVAL_SUMMARY_JSON = perclass_out_dir / "eval_summary.json"
CONFUSION_MATRIX_CSV = perclass_out_dir / "confusion_matrix.csv"
CONFUSION_MATRIX_PNG = perclass_out_dir / "confusion_matrix.png"
PER_CLASS_METRICS_CSV = perclass_out_dir / "per_class_metrics.csv"

# Nếu bạn đã chạy:
# full_eval_result = export_full_detection_eval(...)
# thì lấy summary từ đó. Nếu chưa có thì để None để không lỗi.
try:
    full_eval_summary = full_eval_result.get("summary", None)
except Exception:
    full_eval_summary = None

final_output = {
    "training_config": {
        "run_name": RUN_NAME,
        "model_init": MODEL_INIT,
        "pretrain_weights": PRETRAIN_WEIGHTS,
        "note": "P2 architecture trained for this run, initialized only from base yolo26s.pt weights; no old best.pt fine-tuning. Tested with the new best.pt created by this run.",
        "new_best_model_path": best_model_path,
        "best_model_alias": str(BEST_ALIAS),
        "imgsz": FINAL_IMGSZ,
        "batch": FINAL_BATCH,
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "loss_weights": {
            "box": BOX_GAIN,
            "cls": CLS_GAIN,
            "dfl": DFL_GAIN,
        },
        "optimizer": {
            "name": "AdamW",
            "lr0": 0.001,
            "lrf": 0.01,
            "weight_decay": 0.005,
            "warmup_epochs": 8,
            "cos_lr": True,
        },
        "augmentation": {
            "degrees": 1.0,
            "translate": 0.02,
            "scale": 0.15,
            "mosaic": 0.30,
            "mixup": 0.03,
            "cutmix": 0.05,
            "erasing": 0.0,
            "close_mosaic": 120,
        },
    },

    "dataset_info": {
        "dataset_root": str(dataset_root),
        "data_yaml": str(DATA_YAML),
        "num_classes": len(class_names),
        "class_names": {str(i): name for i, name in enumerate(class_names)},
        "train_image_dir": data_fixed["train"],
        "val_image_dir": data_fixed["val"],
        "test_image_dir": data_fixed["test"],
        "train_images": len(list_images(data_fixed["train"])),
        "val_images": len(list_images(data_fixed["val"])),
        "test_images": len(list_images(data_fixed["test"])),
    },

    "inference_config": {
        "base_conf_before_filter": PRED_BASE_CONF,
        "nms_iou": PRED_NMS_IOU,
        "max_det": MAX_DET,
        "class_conf_thresholds": {
            str(i): {
                "name": class_names[i],
                "conf": CLASS_CONF[i],
            }
            for i in range(len(class_names))
        },
        "eval_iou_threshold": EVAL_IOU_THRESH,
    },

    "ultralytics_test_metrics": ultralytics_metrics,
    "custom_perclass_conf_metrics": overall_custom_metrics,
    "full_detection_eval_metrics": full_eval_summary,
    "per_class_metrics": per_class_metrics,
    "size_based_metrics": size_metrics,
    "inference_speed": speed_info,

    "output_files": {
        "perclass_output_dir": str(perclass_out_dir),

        # File cũ của bạn
        "predictions_perclass_conf": str(perclass_out_dir / "predictions_perclass_conf.json"),
        "records_per_image": str(perclass_out_dir / "records_per_image.json"),

        # File mới bổ sung để đánh giá đầy đủ
        "full_eval_json": str(FULL_EVAL_JSON),
        "eval_summary": str(EVAL_SUMMARY_JSON),
        "confusion_matrix_csv": str(CONFUSION_MATRIX_CSV),
        "confusion_matrix_png": str(CONFUSION_MATRIX_PNG),
        "per_class_metrics_csv": str(PER_CLASS_METRICS_CSV),

        # File tổng
        "final_json": str(FINAL_JSON),
        "history_csv": "/kaggle/working/history.csv",
        "best_model": str(BEST_ALIAS),
    },

    "hardware": {
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    },
}

with open(FINAL_JSON, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)

print("Saved final results to", FINAL_JSON)

ZIP_BASE = f"/kaggle/working/{MODEL_TAG}_train_test_perclass_outputs"

try:
    shutil.make_archive(ZIP_BASE, "zip", perclass_out_dir.parent)
    print("Zipped outputs to", ZIP_BASE + ".zip")
except Exception as e:
    print("Zip failed:", e)

print("\nDONE")
print("New best model:", BEST_ALIAS)
print("Final JSON:", FINAL_JSON)
print("Per-class predictions:", perclass_out_dir / "predictions_perclass_conf.json")
print("Full eval JSON:", FULL_EVAL_JSON)
print("Eval summary:", EVAL_SUMMARY_JSON)
print("Confusion matrix CSV:", CONFUSION_MATRIX_CSV)
print("Confusion matrix PNG:", CONFUSION_MATRIX_PNG)
print("Per-class metrics CSV:", PER_CLASS_METRICS_CSV)